In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [13]:
# Loading the datasets
df_sentiment = pd.read_csv('../data/raw/sentimentdataset.csv')
df_tweets = pd.read_csv('../data/raw/Tweets.csv')

In [15]:
print("=== RAW DATA LOADED ===")
print(f"sentimentdataset.csv  → {df_sentiment.shape[0]} rows, {df_sentiment.shape[1]} columns")
print(f"Tweets.csv            → {df_tweets.shape[0]} rows, {df_tweets.shape[1]} columns")

=== RAW DATA LOADED ===
sentimentdataset.csv  → 732 rows, 15 columns
Tweets.csv            → 14640 rows, 15 columns


In [17]:
print("\nsentimentdataset.csv columns:", df_sentiment.columns.tolist())
print("Tweets.csv columns:", df_tweets.columns.tolist())
 


sentimentdataset.csv columns: ['Unnamed: 0.1', 'Unnamed: 0', 'Text', 'Sentiment', 'Timestamp', 'User', 'Platform', 'Hashtags', 'Retweets', 'Likes', 'Country', 'Year', 'Month', 'Day', 'Hour']
Tweets.csv columns: ['tweet_id', 'airline_sentiment', 'airline_sentiment_confidence', 'negativereason', 'negativereason_confidence', 'airline', 'airline_sentiment_gold', 'name', 'negativereason_gold', 'retweet_count', 'text', 'tweet_coord', 'tweet_created', 'tweet_location', 'user_timezone']


#### ================================================================
#### STEP 1: DROP USELESS COLUMNS — KEEP ONLY text + sentiment
#### ================================================================
 sentimentdataset.csv has 15 columns — we only need Text and Sentiment.

 Tweets.csv has 15 columns — we only need text and airline_sentiment. 
 
 Everything else (timestamps, user IDs, locations etc.) is irrelevant
 for training a sentiment model

In [18]:
df_sentiment = df_sentiment[['Text', 'Sentiment']]
df_tweets = df_tweets[['text', 'airline_sentiment']]

In [20]:
# Rename columns to a consistent standard across both datasets
df_sentiment = df_sentiment.rename(columns={'Text': 'text', 'Sentiment': 'sentiment'})
df_tweets = df_tweets.rename(columns={'airline_sentiment': 'sentiment'})


print("\n=== AFTER DROPPING USELESS COLUMNS ===")
print(f"df_sentiment columns: {df_sentiment.columns.tolist()}")
print(f"df_tweets columns: {df_tweets.columns.tolist()}")


=== AFTER DROPPING USELESS COLUMNS ===
df_sentiment columns: ['text', 'sentiment']
df_tweets columns: ['text', 'sentiment']


In [22]:
print("\n=== NULL VALUES CHECK ===")
print("df_sentiment nulls:\n", df_sentiment.isnull().sum())
print("\ndf_tweets nulls:\n", df_tweets.isnull().sum())


# no null values returned so no need for dropping nas or filling them in
 


=== NULL VALUES CHECK ===
df_sentiment nulls:
 text         0
sentiment    0
dtype: int64

df_tweets nulls:
 text         0
sentiment    0
dtype: int64


In [25]:
# drop rows where text is just whitespace or empty string
df_sentiment = df_sentiment[df_sentiment['text'].str.strip().str.len() > 0]
df_tweets = df_tweets[df_tweets['text'].str.strip().str.len() > 0]


In [26]:
def normalize_label(label):
    """
    Maps any sentiment label to one of three standard classes:
    Positive, Negative, or Neutral.
    Returns None for labels we cannot confidently map.
    """
    label = str(label).strip().lower()
 
    positive_labels = [
        'positive', 'joy', 'happiness', 'happy', 'love', 'excitement',
        'excited', 'contentment', 'content', 'gratitude', 'grateful',
        'serenity', 'serene', 'hopeful', 'hope', 'awe', 'pride',
        'acceptance', 'relief', 'mischievous', 'enthusiasm'
    ]
 
    negative_labels = [
        'negative', 'anger', 'angry', 'fear', 'sadness', 'sad',
        'disgust', 'disgusted', 'grief', 'despair', 'loneliness',
        'lonely', 'embarrassed', 'embarrassment', 'hate', 'bad',
        'nostalgia', 'confusion', 'confused', 'anxiety', 'anxious'
    ]
 
    neutral_labels = [
        'neutral', 'curiosity', 'curious', 'surprise', 'surprised'
    ]
 
    if label in positive_labels:
        return 'Positive'
    elif label in negative_labels:
        return 'Negative'
    elif label in neutral_labels:
        return 'Neutral'
    else:
        # Any label we cannot confidently classify is dropped
        return None
 

In [31]:
# Apply normalisation to both datasets
df_sentiment['sentiment'] = df_sentiment['sentiment'].apply(normalize_label)
df_tweets['sentiment'] = df_tweets['sentiment'].apply(normalize_label)


 

In [32]:
# Drop rows where the label could not be mapped
before_sentiment = len(df_sentiment)
before_tweets = len(df_tweets)
 
df_sentiment = df_sentiment.dropna(subset=['sentiment'])
df_tweets = df_tweets.dropna(subset=['sentiment'])
 
print("\n=== LABEL NORMALISATION ===")
print(f"df_sentiment: {before_sentiment} → {len(df_sentiment)} rows (dropped {before_sentiment - len(df_sentiment)} unmappable labels)")
print(f"df_tweets: {before_tweets} → {len(df_tweets)} rows (dropped {before_tweets - len(df_tweets)} unmappable labels)")
 
print("\ndf_sentiment label distribution after normalisation:")
print(df_sentiment['sentiment'].value_counts())
 
print("\ndf_tweets label distribution after normalisation:")
print(df_tweets['sentiment'].value_counts())
 
 


=== LABEL NORMALISATION ===
df_sentiment: 732 → 386 rows (dropped 346 unmappable labels)
df_tweets: 14640 → 14640 rows (dropped 0 unmappable labels)

df_sentiment label distribution after normalisation:
sentiment
Positive    252
Negative     94
Neutral      40
Name: count, dtype: int64

df_tweets label distribution after normalisation:
sentiment
Negative    9178
Neutral     3099
Positive    2363
Name: count, dtype: int64
